In [1]:
data = {'order_id': 10784, 'customer_id': 8226, 'expected_delivery': '2026-08-20', 'delay_days': 9}

In [2]:
type(data)

dict

In [3]:
# rag

In [4]:
QDRANT_URL: str = "http://localhost:6333"

QDRANT_COLLECTION: str = (
    "customer_operations_knowledge"
)

EMBEDDING_MODEL: str = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [45]:
from langchain_huggingface import (
    HuggingFaceEmbeddings,)

In [46]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6032.95it/s]


In [8]:
from pathlib import Path

KNOWLEDGE_DIR = Path("../data/knowledge")

In [9]:
KNOWLEDGE_DIR

PosixPath('../data/knowledge')

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:



def load_documents() -> list[dict]:
    documents = []

    for path in KNOWLEDGE_DIR.glob("*.txt"):
        text = path.read_text(
            encoding="utf-8"
        )

        documents.append(
            {
                "text": text,
                "source": path.name,
            }
        )

    return documents

In [12]:


def chunk_documents(
    documents: list[dict],
) -> list[dict]:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100,
    )

    chunks = []

    for document in documents:
        split_texts = splitter.split_text(
            document["text"]
        )

        for index, text in enumerate(split_texts):
            chunks.append(
                {
                    "text": text,
                    "source": document["source"],
                    "chunk_index": index,
                }
            )

    return chunks

In [47]:
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient

def create_vector_store(
    texts: list[str],
    metadatas: list[dict],
):
    return QdrantVectorStore.from_texts(
        texts=texts,
        embedding=embeddings,
        metadatas=metadatas,
        url=QDRANT_URL,
        collection_name=QDRANT_COLLECTION,
    )

In [31]:
documents = load_documents()

In [32]:
documents

[{'text': 'ACME COMMERCE SHIPPING POLICY\nVersion: 1.0\n\n1. Standard Delivery\n\nStandard delivery normally takes 3–5 business days.\n\n2. Delayed Shipments\n\nAn order is considered delayed when the current date is later than\nthe expected delivery date.\n\nIf an order is delayed by more than 48 hours:\n\n- The customer should be notified.\n- The shipment tracking information should be reviewed.\n- If tracking has not updated for more than 48 hours, the carrier\n  should be contacted.\n\n3. Serious Delays\n\nIf an order is delayed by more than 5 days:\n\n- The case should be treated as high priority.\n- The shipment should be investigated.\n- A support ticket should be created.\n- The customer should be notified.\n\n4. Missing Tracking Information\n\nIf an order is delayed but has no tracking number:\n\n- Do not assume the shipment was lost.\n- Verify the order and shipment records.\n- Escalate the case to a support agent.\n\n5. Lost Packages\n\nA package may be considered potentiall

In [33]:
chunks = chunk_documents(documents)

In [34]:
chunks

[{'text': 'ACME COMMERCE SHIPPING POLICY\nVersion: 1.0\n\n1. Standard Delivery\n\nStandard delivery normally takes 3–5 business days.\n\n2. Delayed Shipments\n\nAn order is considered delayed when the current date is later than\nthe expected delivery date.\n\nIf an order is delayed by more than 48 hours:\n\n- The customer should be notified.\n- The shipment tracking information should be reviewed.\n- If tracking has not updated for more than 48 hours, the carrier\n  should be contacted.\n\n3. Serious Delays\n\nIf an order is delayed by more than 5 days:\n\n- The case should be treated as high priority.\n- The shipment should be investigated.\n- A support ticket should be created.\n- The customer should be notified.',
  'source': 'shipping_policy.txt',
  'chunk_index': 0},
 {'text': '4. Missing Tracking Information\n\nIf an order is delayed but has no tracking number:\n\n- Do not assume the shipment was lost.\n- Verify the order and shipment records.\n- Escalate the case to a support ag

In [35]:
chunks[0]

{'text': 'ACME COMMERCE SHIPPING POLICY\nVersion: 1.0\n\n1. Standard Delivery\n\nStandard delivery normally takes 3–5 business days.\n\n2. Delayed Shipments\n\nAn order is considered delayed when the current date is later than\nthe expected delivery date.\n\nIf an order is delayed by more than 48 hours:\n\n- The customer should be notified.\n- The shipment tracking information should be reviewed.\n- If tracking has not updated for more than 48 hours, the carrier\n  should be contacted.\n\n3. Serious Delays\n\nIf an order is delayed by more than 5 days:\n\n- The case should be treated as high priority.\n- The shipment should be investigated.\n- A support ticket should be created.\n- The customer should be notified.',
 'source': 'shipping_policy.txt',
 'chunk_index': 0}

In [36]:
chunks[0]["text"]

'ACME COMMERCE SHIPPING POLICY\nVersion: 1.0\n\n1. Standard Delivery\n\nStandard delivery normally takes 3–5 business days.\n\n2. Delayed Shipments\n\nAn order is considered delayed when the current date is later than\nthe expected delivery date.\n\nIf an order is delayed by more than 48 hours:\n\n- The customer should be notified.\n- The shipment tracking information should be reviewed.\n- If tracking has not updated for more than 48 hours, the carrier\n  should be contacted.\n\n3. Serious Delays\n\nIf an order is delayed by more than 5 days:\n\n- The case should be treated as high priority.\n- The shipment should be investigated.\n- A support ticket should be created.\n- The customer should be notified.'

In [37]:
texts = [chunk["text"] for chunk in chunks]

In [38]:
texts

['ACME COMMERCE SHIPPING POLICY\nVersion: 1.0\n\n1. Standard Delivery\n\nStandard delivery normally takes 3–5 business days.\n\n2. Delayed Shipments\n\nAn order is considered delayed when the current date is later than\nthe expected delivery date.\n\nIf an order is delayed by more than 48 hours:\n\n- The customer should be notified.\n- The shipment tracking information should be reviewed.\n- If tracking has not updated for more than 48 hours, the carrier\n  should be contacted.\n\n3. Serious Delays\n\nIf an order is delayed by more than 5 days:\n\n- The case should be treated as high priority.\n- The shipment should be investigated.\n- A support ticket should be created.\n- The customer should be notified.',
 '4. Missing Tracking Information\n\nIf an order is delayed but has no tracking number:\n\n- Do not assume the shipment was lost.\n- Verify the order and shipment records.\n- Escalate the case to a support agent.\n\n5. Lost Packages\n\nA package may be considered potentially lost w

In [39]:
metadatas = [
        {
            "source": chunk["source"],
            "chunk_index": chunk["chunk_index"],
        }
        for chunk in chunks
    ]

In [49]:
# create_vector_store(texts=texts, metadatas=metadatas)

In [50]:
from langchain_qdrant import QdrantVectorStore

In [51]:
vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    url=QDRANT_URL,
    collection_name=QDRANT_COLLECTION,
)

In [52]:
def search_policy(
    query: str,
    limit: int = 5,
) -> list[dict]:

    results = vector_store.similarity_search(
        query,
        k=limit,
    )

    return [
        {
            "content": document.page_content,
            "source": document.metadata.get(
                "source"
            ),
            "chunk_index": document.metadata.get(
                "chunk_index"
            ),
        }
        for document in results
    ]

In [53]:
search_policy("What happens when an order is delayed more than 10 days?")

[{'content': 'ACME COMMERCE SHIPPING POLICY\nVersion: 1.0\n\n1. Standard Delivery\n\nStandard delivery normally takes 3–5 business days.\n\n2. Delayed Shipments\n\nAn order is considered delayed when the current date is later than\nthe expected delivery date.\n\nIf an order is delayed by more than 48 hours:\n\n- The customer should be notified.\n- The shipment tracking information should be reviewed.\n- If tracking has not updated for more than 48 hours, the carrier\n  should be contacted.\n\n3. Serious Delays\n\nIf an order is delayed by more than 5 days:\n\n- The case should be treated as high priority.\n- The shipment should be investigated.\n- A support ticket should be created.\n- The customer should be notified.',
  'source': 'shipping_policy.txt',
  'chunk_index': 0},
 {'content': 'ACME COMMERCE SHIPPING POLICY\nVersion: 1.0\n\n1. Standard Delivery\n\nStandard delivery normally takes 3–5 business days.\n\n2. Delayed Shipments\n\nAn order is considered delayed when the current da